## 1. get model and embedder ready

In [1]:
!wget -O src/embeddings/download.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/download.py

--2026-08-12 16:31:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘src/embeddings/download.py’

src/embeddings/down 100%[===================>]   1.34K  --.-KB/s    in 0.05s   

2026-08-12 16:31:34 (25.7 KB/s) - ‘src/embeddings/download.py’ saved [1376/1376]



In [2]:
!uv run python src/embeddings/download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [3]:
!wget -O src/embeddings/embedder.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/embedder.py

--2026-08-12 16:31:55--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘src/embeddings/embedder.py’

src/embeddings/embe 100%[===================>]   1.48K  --.-KB/s    in 0.02s   

2026-08-12 16:31:56 (64.6 KB/s) - ‘src/embeddings/embedder.py’ saved [1520/1520]



In [7]:
from pathlib import Path

print("cwd:", Path.cwd())

cwd: /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens


In [8]:
path = Path("models/Xenova/all-MiniLM-L6-v2")
print("resolved:", path.resolve())

resolved: /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/models/Xenova/all-MiniLM-L6-v2


In [9]:
print("directory exists:", path.exists())

directory exists: True


In [10]:
print("tokenizer exists:", (path / "tokenizer.json").exists())

tokenizer exists: True


In [14]:
from src.embeddings.embedder import Embedder

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

---

## 2. in memory Vector Search

In [1]:
# autoreload every imported modules if their source files have changed.
%load_ext autoreload
%autoreload 2

In [2]:
from src.ingest_lyrics import load_lyrics

lyrics_df = load_lyrics()
lyrics_df

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
0,$ex Appeal,Baby Keem Featuring Too $hort,[2026-03-07],1,69,"(Play with me, play with me)\n(Play with me, p..."
1,'98 Braves,Morgan Wallen,"[2023-03-18, 2023-03-25, 2023-04-01, 2023-04-0...",7,27,I remember sittin' at that house\nLivin' room ...
2,'Til You Can't,Cody Johnson,"[2021-10-23, 2021-10-30, 2021-11-06, 2022-01-0...",29,18,You can tell your old man\nYou'll do some larg...
3,'Tis The Damn Season,Taylor Swift,"[2020-12-26, 2021-01-02]",2,39,If I wanted to know who you were hanging with\...
4,(There's No Place Like) Home For The Holidays ...,Perry Como With Mitchell Ayers And His Orchestra,[2024-01-06],1,50,"Oh, there's no place like home for the holiday..."
...,...,...,...,...,...,...
4079,pov,Ariana Grande,"[2020-11-14, 2020-11-21, 2020-11-28, 2020-12-0...",20,27,It's like you got superpowers\nTurn my minutes...
4080,punchin'.the.clock,J. Cole,"[2021-05-29, 2021-06-05]",2,20,It ain't nothin' I want more\nAin't nothin' I ...
4081,thanK you aIMee,Taylor Swift,"[2024-05-04, 2024-05-11, 2024-05-18]",3,23,"When I picture my hometown\nThere's a bronze, ..."
4082,the.climb.back,J. Cole,"[2021-05-29, 2021-06-05]",2,25,Are you doin' this work to facilitate growth o...


In [3]:
import numpy as np

from src.embeddings.download import download
from src.embeddings.embedder import Embedder
from src.embeddings.embed_texts import embed_texts

from src.ingest_lyrics import load_lyrics
from src.chunk_lyrics import build_chunk_documents
from src.config import MODEL_NAME, MODEL_PATH

download(MODEL_NAME)
embedder = Embedder(MODEL_PATH)

lyrics_df = load_lyrics()
documents = build_chunk_documents(lyrics_df)

texts = [doc["section"] for doc in documents]
vectors = embed_texts(texts, embedder)
X = np.array(vectors)

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx
0 problems occured when chunking lyrics.
41681 lyrics chunks generated.


  0%|          | 0/834 [00:00<?, ?it/s]

In [4]:
documents[0]

{'df_song_id': 0,
 'title': '$ex Appeal',
 'performer': 'Baby Keem Featuring Too $hort',
 'section_id': 1,
 'section': "(Play with me, play with me)\n(Play with me, pla-play with me) up all night, baby\n(Play with me, play with me)\n(Play with me, pla-play with me) it's $hort Dog",
 'num_lines': 4}

In [5]:
query = "Grief after losing best friend"
v_query = embedder.encode(query)

scores = X.dot(v_query)
top5 = np.argsort(scores)[-5:][::-1]

for i in top5:
    print(scores[i])
    print(documents[i])

0.5263760866097053
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 7, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think that this was best left\nThere in the past where it belongs\nI feel an overwhelming sadness\nOf all the friends I do not have left\nSeeing how my family has fractured\nGrowin' up and movin' on", 'num_lines': 8}
0.5258248721371901
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 2, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think that this was best left\nIn the past where it belongs\nI feel an overwhelming sadness\nOf all the friends I do not have left\nSeeing how my family has fractured\nGrowin' up and movin' on", 'num_lines': 8}
0.522244341255196
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 4, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think 

---

## 3. PostgreSQL Vector Search